# Pink noise from white noise via a fractional integral

Power spectrum $S(f)\propto 1/f^{\beta}$. A fractional integral of order $\gamma=\beta/2$ applied to white noise
multiplies the amplitude spectrum by $f^{-\beta/2}$ and hence the power spectrum by $f^{-\beta}$.

Three methods:
1. **FFT**: scale the spectrum by $f^{-\beta/2}$, set DC to zero, inverse FFT.
2. **Grünwald–Letnikov**: weights $w_0=1,\ w_k=w_{k-1}\,(k-1+\gamma)/k$, $x_n=\sum_k w_k\,\xi_{n-k}$ via `fftconvolve`.
3. **Sum of first-order filters** $y_n=a\,y_{n-1}+b\,\xi_n$ (the Kellett idea), via `lfilter`.

For each: Welch spectrum, log-log plot, slope fit, runtime. Then 5 s wav files (white / pink / Brownian).

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from scipy.optimize import least_squares

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

OUT = Path(r"C:\Noises") if Path(r"C:\Noises").exists() else Path(".")
(OUT / "audio").mkdir(exist_ok=True)

N      = 2**16
FS     = 44100
BETAS  = [0.5, 1.0, 1.5]
rng    = np.random.default_rng(42)
white  = rng.standard_normal(N)

# range used for the slope fit (avoid the very lowest bins and the filter-edge region near Nyquist)
FIT_LO, FIT_HI = 100.0, 10_000.0

## Method 1: FFT

In [ ]:
def pink_fft(xi, beta):
    n = len(xi)
    X = np.fft.rfft(xi)
    f = np.fft.rfftfreq(n, d=1/FS)
    H = np.zeros_like(f)
    H[1:] = f[1:] ** (-beta / 2)          # zero frequency -> 0
    return np.fft.irfft(X * H, n=n)

## Method 2: Grünwald–Letnikov

In [ ]:
def gl_weights(gamma, n):
    k = np.arange(1, n)
    w = np.empty(n)
    w[0] = 1.0
    w[1:] = np.cumprod((k - 1 + gamma) / k)   # w_k = w_{k-1} (k-1+gamma)/k
    return w

def pink_gl(xi, beta):
    gamma = beta / 2
    w = gl_weights(gamma, len(xi))
    return signal.fftconvolve(xi, w)[: len(xi)]

## Method 3: sum of first-order filters

Each section is $y_n=a_k y_{n-1}+b_k\xi_n$ with the pole $a_k=e^{-2\pi f_k/f_s}$ on a logarithmic grid of corner frequencies $f_k$.
The gains $b_k$ are fitted so that $|\sum_k H_k(f)|\approx f^{-\beta/2}$ over 20 Hz – 15 kHz (least squares in the log domain).

In [ ]:
def design_exp_bank(beta, n_sections=6, f_lo=20.0, f_hi=15_000.0, fs=FS):
    fk = np.geomspace(f_lo, f_hi, n_sections)             # log grid of corner frequencies
    a  = np.exp(-2 * np.pi * fk / fs)
    fg = np.geomspace(f_lo, f_hi, 400)
    z  = np.exp(-2j * np.pi * fg / fs)
    Hk = 1.0 / (1.0 - a[:, None] * z[None, :])             # response of y_n = a y_{n-1} + x_n
    target = np.log(fg ** (-beta / 2))

    def resid(logc):
        H = (np.exp(logc)[:, None] * Hk).sum(axis=0)
        e = np.log(np.abs(H)) - target
        return e - e.mean()                                # overall level is arbitrary

    sol = least_squares(resid, np.zeros(n_sections) - np.log(a) * 0 )
    b = np.exp(sol.x)
    return a, b

def pink_exp(xi, a, b):
    return sum(signal.lfilter([bk], [1.0, -ak], xi) for ak, bk in zip(a, b))

### Kellett filter for comparison

In [ ]:
# Paul Kellett's "refined" pink filter (musicdsp.org), designed for fs = 44.1 kHz.
# Coefficients typed from memory of the musicdsp.org post (no network here) - please double-check them there.
KELLETT_A = np.array([0.99886, 0.99332, 0.96900, 0.86650, 0.55000, -0.76160])
KELLETT_B = np.array([0.0555179, 0.0750759, 0.1538520, 0.3104856, 0.5329522, -0.0168980])
KELLETT_DIRECT = 0.5362      # white*0.5362 added directly
KELLETT_B6 = 0.115926        # b6 = white*0.115926 (one-sample delayed in the original code)

def pink_kellett(xi):
    y = sum(signal.lfilter([bk], [1.0, -ak], xi) for ak, bk in zip(KELLETT_A, KELLETT_B))
    y6 = np.concatenate(([0.0], xi[:-1])) * KELLETT_B6
    return y + y6 + xi * KELLETT_DIRECT

pos = KELLETT_A > 0
print("Kellett positive poles -> corner frequencies, Hz:",
      np.round(-FS * np.log(KELLETT_A[pos]) / (2 * np.pi), 1))
print("(the pole at -0.7616 is a high-frequency one, near Nyquist)")

## Spectrum and slope

In [ ]:
def spectrum(x, nperseg=4096):
    f, S = signal.welch(x, fs=FS, nperseg=nperseg, window="hann", detrend="constant")
    return f[1:], S[1:]

def fit_slope(f, S, lo=FIT_LO, hi=FIT_HI):
    # resample onto a log-uniform grid so every decade carries equal weight
    fg = np.geomspace(lo, hi, 200)
    Sg = np.interp(fg, f, S)
    return np.polyfit(np.log10(fg), np.log10(Sg), 1)[0]

def timeit(fn, repeat=5):
    best = np.inf
    for _ in range(repeat):
        t = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t)
    return out, best * 1e3        # ms

In [ ]:
results, signals = [], {}
design_ms = {}

for beta in BETAS:
    t = time.perf_counter()
    a, b = design_exp_bank(beta)
    design_ms[beta] = (time.perf_counter() - t) * 1e3

    methods = {
        "FFT":          lambda: pink_fft(white, beta),
        "Grünwald–L.":  lambda: pink_gl(white, beta),
        "Exp. sum":     lambda: pink_exp(white, a, b),
    }
    for name, fn in methods.items():
        x, ms = timeit(fn)
        f, S = spectrum(x)
        slope = fit_slope(f, S)
        signals[(beta, name)] = (f, S)
        results.append((beta, name, slope, ms))

print(f"{'target beta':>11} | {'method':<12} | {'slope':>7} | {'error':>7} | {'time, ms':>8}")
print("-" * 58)
for beta, name, slope, ms in results:
    print(f"{beta:>11.1f} | {name:<12} | {slope:>7.3f} | {slope + beta:>+7.3f} | {ms:>8.2f}")
print("\nExp. sum design (least squares) time, ms:", {k: round(v, 1) for k, v in design_ms.items()})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharey=False)
colors = {"FFT": "C0", "Grünwald–L.": "C1", "Exp. sum": "C2"}
for ax, beta in zip(axes, BETAS):
    for name in colors:
        f, S = signals[(beta, name)]
        ax.loglog(f, S, color=colors[name], alpha=0.8, lw=1.2, label=name)
    f, S = signals[(beta, "FFT")]
    ref = S[np.argmin(abs(f - 1000))] * (f / 1000) ** (-beta)
    ax.loglog(f, ref, "k--", lw=1, label=rf"ideal $1/f^{{{beta}}}$")
    ax.axvspan(FIT_LO, FIT_HI, color="gray", alpha=0.08)
    ax.set(title=rf"$\beta={beta}$", xlabel="f, Hz", ylabel="PSD")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Kellett vs. our fitted bank (β = 1)

In [ ]:
xk = pink_kellett(white)
fk, Sk = spectrum(xk)
a1, b1 = design_exp_bank(1.0)
f1, S1 = signals[(1.0, "Exp. sum")]

print("Kellett slope:", round(fit_slope(fk, Sk), 3))
print("Our corner frequencies, Hz:", np.round(-FS * np.log(a1) / (2 * np.pi), 1))

# deviation from an ideal 1/f line (dB), normalised at 1 kHz
def dev_db(f, S):
    ref = S[np.argmin(abs(f - 1000))] * (f / 1000) ** -1.0
    return 10 * np.log10(S / ref)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(fk, Sk, label="Kellett"); ax[0].loglog(f1, S1, label="fitted bank")
ax[0].set(xlabel="f, Hz", ylabel="PSD", title="Spectra"); ax[0].legend()
ax[1].semilogx(fk, dev_db(fk, Sk), label="Kellett"); ax[1].semilogx(f1, dev_db(f1, S1), label="fitted bank")
ax[1].set(xlabel="f, Hz", ylabel="deviation from 1/f, dB", ylim=(-6, 6), title="Deviation (Welch, noisy)"); ax[1].legend()
plt.tight_layout(); plt.show()

## Audio: white, pink, Brownian (5 s, 44.1 kHz)

In [ ]:
n_audio = 5 * FS
xi_a = np.random.default_rng(7).standard_normal(n_audio)

audio = {
    "white":    pink_fft(xi_a, 0.0),   # beta = 0 -> unchanged apart from DC removal
    "pink":     pink_fft(xi_a, 1.0),
    "brownian": pink_fft(xi_a, 2.0),
}

def to_int16(x, rms=0.1, peak_limit=0.95):
    x = x - x.mean()
    x = x * (rms / x.std())                      # equal RMS => comparable loudness
    p = np.abs(x).max()
    if p > peak_limit:
        x *= peak_limit / p; print(f"  peak-limited (peak was {p:.2f})")
    return (x * 32767).astype(np.int16)

for name, x in audio.items():
    wavfile.write(OUT / "audio" / f"{name}.wav", FS, to_int16(x))
    f, S = spectrum(x)
    print(f"{name:9s} slope = {fit_slope(f, S):+.3f}")
print("Saved to", (OUT / "audio").resolve())

fig, ax = plt.subplots(figsize=(7, 4))
for name, x in audio.items():
    f, S = spectrum(x); ax.loglog(f, S, label=name)
ax.set(xlabel="f, Hz", ylabel="PSD", title="Audio spectra"); ax.legend(); plt.show()

try:
    from IPython.display import Audio, display
    for name, x in audio.items():
        print(name); display(Audio(to_int16(x) / 32768, rate=FS))
except Exception:
    pass

## Notes

* **FFT** is exact in the frequency domain but treats the signal as periodic (circular), so it has no transient and the best slope.
* **Grünwald–Letnikov** weights decay as $k^{\gamma-1}$, so with a finite record the start of the signal lacks "memory"; the effect is largest for big $\beta$. A burn-in (generate longer, drop the head) removes it.
* **Exponential sum** is a cheap IIR (real-time friendly) but only approximates the slope inside its design band (ripple of a fraction of a dB with 6 sections).
* Brownian noise ($\beta=2$) is the case $\gamma=1$, the plain integral (running sum).